# VibeMatch — Dataset Exploration

EDA for the Movie Posters + Book Covers datasets.

**Run order:** `scripts/download_data.py` → `scripts/preprocess.py` → this notebook.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from src.loaders.dataset import parse_genres
from src.loaders.split import build_genre_vocab, genre_stratified_split

PROCESSED = REPO_ROOT / "data" / "processed"
movies = pd.read_csv(PROCESSED / "movies.csv")
books = pd.read_csv(PROCESSED / "books.csv")
df = pd.concat([movies, books], ignore_index=True)

print(f"movies: {len(movies)}")
print(f"books:  {len(books)}")
print(f"total:  {len(df)}")

## 1. Genre frequency

In [ ]:
genres_per_item = [parse_genres(g) for g in df["genres"]]
from collections import Counter
counts = Counter(g for gs in genres_per_item for g in gs)

top = counts.most_common(20)
labels, values = zip(*top)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(labels, values)
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_ylabel("# items")
ax.set_title("Top 20 genres (movies + books combined)")
fig.tight_layout()
plt.show()

print(f"unique genres: {len(counts)}")
print(f"avg genres / item: {np.mean([len(g) for g in genres_per_item]):.2f}")

## 2. Image size distribution

In [ ]:
sample = df.sample(min(500, len(df)), random_state=0)
sizes = []
for p in sample["image_path"]:
    try:
        with Image.open(REPO_ROOT / p) as im:
            sizes.append(im.size)  # (W, H)
    except Exception as e:
        print(f"skipped {p}: {e}")

widths, heights = zip(*sizes)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(widths, bins=30)
axes[0].set_title("Width")
axes[1].hist(heights, bins=30)
axes[1].set_title("Height")
fig.tight_layout()
plt.show()

print(f"width  median {int(np.median(widths))}, p5 {int(np.percentile(widths, 5))}, p95 {int(np.percentile(widths, 95))}")
print(f"height median {int(np.median(heights))}, p5 {int(np.percentile(heights, 5))}, p95 {int(np.percentile(heights, 95))}")

## 3. Sample grid by genre

In [ ]:
def show_grid(genre, n=8):
    has_genre = df[df["genres"].str.contains(genre, na=False, regex=False)]
    if len(has_genre) == 0:
        print(f"no items for {genre}")
        return
    sub = has_genre.sample(min(n, len(has_genre)), random_state=0)
    cols = min(n, 4)
    rows = (len(sub) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 3.5))
    axes = np.array(axes).reshape(-1)
    for ax, (_, row) in zip(axes, sub.iterrows()):
        try:
            ax.imshow(Image.open(REPO_ROOT / row["image_path"]))
        except Exception:
            pass
        ax.set_title(row["title"][:30] if isinstance(row["title"], str) else "", fontsize=8)
        ax.axis("off")
    for ax in axes[len(sub):]:
        ax.axis("off")
    fig.suptitle(genre)
    fig.tight_layout()
    plt.show()

for g, _ in counts.most_common(4):
    show_grid(g, n=8)

## 4. Genre-stratified split balance

Confirms the rare-genre stratification keeps tail genres represented in every split.

In [ ]:
vocab = build_genre_vocab(genres_per_item, min_count=5)
split = genre_stratified_split(genres_per_item, val_frac=0.1, test_frac=0.1, seed=42)

def per_split_counts(idx):
    c = Counter()
    for i in idx:
        c.update(genres_per_item[i])
    return c

tr, va, te = per_split_counts(split.train), per_split_counts(split.val), per_split_counts(split.test)
rows = []
for g in vocab:
    rows.append({"genre": g, "train": tr[g], "val": va[g], "test": te[g]})
balance = pd.DataFrame(rows).sort_values("train", ascending=False)
balance.head(20)

In [ ]:
print(f"train {len(split.train)}, val {len(split.val)}, test {len(split.test)}")
print(f"vocab size (min_count=5): {len(vocab)}")
missing_in_val = balance[balance["val"] == 0]["genre"].tolist()
missing_in_test = balance[balance["test"] == 0]["genre"].tolist()
print(f"genres missing in val:  {missing_in_val}")
print(f"genres missing in test: {missing_in_test}")

## 5. Title length / text quality

Sanity check that titles are usable as fallback retrieval text if needed.

In [ ]:
title_lens = df["title"].fillna("").str.split().str.len()
print(f"title word count: median {int(title_lens.median())}, p95 {int(title_lens.quantile(0.95))}")
print(f"empty titles: {(title_lens == 0).sum()}")